In [0]:
from pyspark.sql.types import StructType, StructField, StringType
import requests

user_ids = [
    "211941127362707456",  # Me (ThePriestinator)
    "266038598594195456"   # Additional user
]
sport = "nfl"
league_names = ["League of Inches", "Orange Empire", "(JAIL) Justified Alliance of Ingorant Longshots", "Gold Standard Dynasty"]  # The leagues that I care about.
all_leagues = []

for user_id in user_ids:
    season = 2019
    while True:
        url = (
            f"https://api.sleeper.app/v1/user/{user_id}/leagues/{sport}/{season}"
        )
        response = requests.get(url)
        if response.status_code != 200:
            break
        leagues = response.json()
        if not leagues:
            break
        for league in leagues:
            if league["name"] in league_names:
                all_leagues.append(league)
        season += 1

flat_leagues = [
    {
        "name": league["name"],
        "season": league["season"],
        "league_id": league["league_id"]
    }
    for league in all_leagues
]

schema = StructType([
    StructField("name", StringType(), True),
    StructField("season", StringType(), True),
    StructField("league_id", StringType(), True)
])

df = spark.createDataFrame(flat_leagues, schema)
df.write \
  .mode("overwrite") \
  .saveAsTable("workspace.sleeper_raw.leagues")